In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
data = np.load('L2_Manifold_new.npy', allow_pickle=True)
obj = data.item() 
x1 = obj['x1']
v2 = obj['v2']
T = obj['T']
C = obj['C']
mu = obj['mu']
L1 =  0.836915131427382
L2 =  1.155682161024677
L3 = -1.005062645331442

In [ ]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import pysr 
from pysr import PySRRegressor
from scipy.optimize import least_squares
from sympy.utilities.codegen import codegen
from scipy.integrate import solve_ivp

In [ ]:
def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt

In [ ]:
fig_man, ax_man = plt.subplots(1)
fig_res, ax_res = plt.subplots(1)
for i in range(x1.size):
    #print(i)
    p0 = np.array([x1[i], 0, 0, v2[i]])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T[i]),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T[i], 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:])
    ax_res.plot(x1[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_man.axis('equal')
plt.grid()
plt.show()

In [ ]:
a

In [ ]:
# Считаем первую и вторую производные по a
a = x1-L2
dv2_da = np.gradient(v2, a)
d2v2_da2 = np.gradient(dv2_da, a)

def find_zeros(x, y):
    """Находит x-координаты нулей функции y(x) методом линейной интерполяции."""
    sign_change = np.where(np.diff(np.sign(y)))[0]
    zeros = []
    for i in sign_change:
        x0, x1 = x[i], x[i+1]
        y0, y1 = y[i], y[i+1]
        # Линейная интерполяция для точного положения нуля
        zero_x = x0 - y0 * (x1 - x0) / (y1 - y0)
        zeros.append(zero_x)
    return np.array(zeros)

# Находим нули производных
zeros_first = find_zeros(a, dv2_da)
zeros_second = find_zeros(a, d2v2_da2)

# Для отображения на графике получаем соответствующие значения v2 в этих точках
v2_at_zeros_first = np.interp(zeros_first, a, v2)
v2_at_zeros_second = np.interp(zeros_second, a, v2)

# Строим график
plt.figure(figsize=(10, 6))
plt.plot(a, v2, label="v2", color='blue')
plt.scatter(zeros_first, v2_at_zeros_first, color='red', marker='o', label="Нули 1-й производной")
plt.scatter(zeros_second, v2_at_zeros_second, color='green', marker='x', label="Нули 2-й производной")
plt.xlabel('a')
plt.ylabel('v2')
plt.legend()
plt.title("График функции v2 и нули её производных")
plt.grid(True)
plt.show()

In [ ]:
L2+np.array(zeros_second)

In [ ]:

plt.plot(a, dv2_da, label="v2", color='blue')

In [ ]:
from sympy.utilities.lambdify import lambdastr
from sympy.abc import x

In [ ]:
mu

In [ ]:
np.max(x1-L2)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import solve_ivp
import math

def odeCR3BP(t, p, mu):
    R = np.sqrt((p[0] + mu)**2 + p[1]**2)
    r = np.sqrt((p[0] - 1 + mu)**2 + p[1]**2)
    dpdt = np.array([p[2], p[3],
    2 * p[3] + p[0] - (1 - mu) * (p[0] + mu) / R**3 - mu * (p[0] - 1 + mu) / r**3,
    -2 * p[2] + p[1] - (1 - mu) * p[1] / R**3 - mu * p[1] / r**3])

    return dpdt


x1_fun = lambda x: x
x2_fun = lambda x: 0

def v1_fun(x):
    return 0
def v2_fun(x):
    #y = (-1.4244981*x + 1.2688117 + 0.076973945/math.log(x**2*((x - 0.20886853)**112.4146 + math.sin(x)))) #6.0e-4
    x = x - L2
    if x>=0 and x<0.0394623:
        #y = (x/((2*x*math.atan(math.atan(7.819647*math.sin(math.log(x)) + 0.120930105) - 0.58943594)*math.atan(math.sin(math.log(x))) - x + 0.5879097)*math.log(x + 0.7306801))) #3.8e-05 #(9.999999999998899e-05,0.046486726030323444)
        #y = (-math.atan(2*x + 8.28270529720417*x/((x*math.log(0.857882793307553*x) + x/(2*x - 0.11087902))/math.sqrt((x*math.log(0.857882793307553*x) + x/(2*x - 0.11087902))**2 + 1) + 2.4157221))) #3.5e-05 #(9.999999999998899e-05,0.040894726030323625)
        y = (x*math.exp((x*(x*(math.cos(0.44315043/(x - 0.0712678)) + 2.062117) + 0.46996135))**x)/(4*x - 0.49699014)) #3.0e-05 #(9.999999999998899e-05,0.040894726030323625)

    elif x>=0.0394623 and x<0.24305185:
        y = ((-1.46250174055988*x**3 - 0.35126773*x*(x + 2.807831e-5*math.exp(math.sin((0.38976994*x - 0.7654842)/x)) + 0.013415855) + 9.8630042159363e-6*math.exp(math.sin(0.33920822/x)) + 0.000373935844532679)/x**2) #3.4e-05
    elif x>=0.24305185:
        y = (-1.4388031*x - math.cos(x**2*(1.4388031*math.atan(x) + 0.3300813)*math.atan((x - 0.3300813)*(0.9308737*math.atan(x) + 1.79023032846377)*math.log(x))) + 0.6302827) #1.1e-07
    return y
def T_fun(x):
    #y = (3.24007847470066*x*(-x + math.cos(0.50519687/(x - 0.9149853)) + 2.683576) - x + math.log(1.3429124*x**2 + math.cos(0.37827596/(x - 1.0821371)))*math.sin(1)) #1.4e-2
    x = x - L2
    y = (1.0*x + 0.40856314*math.tanh(math.tanh(1.9805193 - 0.11379527/x)) + 3.683467 + 7.02544599473763*math.exp(-0.17211303/math.tanh(x))) #2.8e-03
    return y
#x_range = np.linspace(np.min(x1), np.max(x1), 100)
x_range = L2+np.linspace(1e-4, 0.3464867, 10)
fig_man, ax_man = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_man.plot(p.y[0,:], p.y[1,:], 'k')
    #ax_res.plot(x_range[i], np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_man.plot(L2, 0, '*k')
ax_man.grid()
#ax_res.grid()

ax_man.set_xlabel('x, безразм.', fontsize=11)
ax_man.set_ylabel('y, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L2_orbits.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')


In [ ]:

x_range = L2+np.linspace(1e-4, 0.3464867, 100)
#x_range = L1-np.linspace(0.02101619, 0.11879741, 100)
fig_res, ax_res = plt.subplots(1)
for i in range(x_range.size):
    #print(i)
    p0 = np.array([x1_fun(x_range[i]), x2_fun(x_range[i]), v1_fun(x_range[i]), v2_fun(x_range[i])])
    p = solve_ivp(lambda t, p: odeCR3BP(t, p, mu),
                  t_span=(0, T_fun(x_range[i])),
                  y0=p0,
                  method='LSODA',
                  t_eval = np.linspace(0, T_fun(x_range[i]), 1000),
                  atol=1e-12,
                  rtol=1e-12)
    ax_res.plot(x_range[i]-L2, np.linalg.norm(p.y[:,-1] - p0), '.k')
ax_res.grid()
ax_res.set_xlabel(r'$x-L_2$, безразм.', fontsize=11)
ax_res.set_ylabel(r'Невязка через период, безразм.', fontsize=11)
# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L2_err.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

In [ ]:

a = x1-L2
trues = v2
preds = [v2_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x-L_2$, безразм.')
ax.set_ylabel(r'$v_y$, безразм.')
ax.legend()

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x-L_2$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L2_vy.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
a = x1-L2
trues = T
preds = [T_fun(x) for x in x1]
preds = np.array(preds).flatten()

# Основной график
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(a, trues, label='Истинное значение', lw=2)
ax.plot(a, preds, '--', label='Аппроксимация', lw=2)
ax.grid()
ax.set_xlabel(r'$x-L_2$, безразм.')
ax.set_ylabel(r'$T$, безразм.')
ax.legend()

# Вложенный график (разница)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
axins = inset_axes(ax, width="40%", height="30%", loc='center right')
axins.plot(a, np.array(trues) - np.array(preds), color='black')
axins.set_title('Разница')
axins.grid()
axins.tick_params(labelsize=8)
axins.set_xlabel(r'$x-L_2$, безразм.', fontsize=8)
axins.set_ylabel('Δ, безразм.', fontsize=8)

# Сохранение с высоким разрешением
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)
figname = 'L2_T.png'
plt.savefig(figures_dir / figname, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(x*math.exp((x*(x*(math.cos(0.44315043/(x - 0.0712678)) + 2.062117) + 0.46996135))**x)/(4*x - 0.49699014)) #3.0e-05"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=5, litera=litera)
latex = latex.replace('x','q')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "((-1.46250174055988*x**3 - 0.35126773*x*(x + 2.807831e-5*math.exp(math.sin((0.38976994*x - 0.7654842)/x)) + 0.013415855) + 9.8630042159363e-6*math.exp(math.sin(0.33920822/x)) + 0.000373935844532679)/x**2) #3.4e-05"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=6, litera=litera)
latex = latex.replace('x','q')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(-1.4388031*x - math.cos(x**2*(1.4388031*math.atan(x) + 0.3300813)*math.atan((x - 0.3300813)*(0.9308737*math.atan(x) + 1.79023032846377)*math.log(x))) + 0.6302827)"

litera = "C"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="v_y", j=7, litera=litera)
latex = latex.replace('x','q')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)

In [ ]:
from ConvertFormula import expr_to_latex_with_consts
expr_str = "(1.0*x + 0.40856314*math.tanh(math.tanh(1.9805193 - 0.11379527/x)) + 3.683467 + 7.02544599473763*math.exp(-0.17211303/math.tanh(x))) #2.8e-03"

litera = "D"
latex, consts = expr_to_latex_with_consts(expr_str, lhs="T", j=3, litera=litera)
latex = latex.replace('x','q')
print(latex)
print("Константы:")
#for i, v in consts:
#    print(f"{litera}_{i}^{{1}} = {v}")
for i, v in consts:
    print(v)